# LP: which loans to fund

New notebook. The ML notebook produced a PD for every loan in the test pool. This picks which of those loans to fund.

**The question.** Given a fixed budget and 409,857 applicants, which loans maximize expected return without taking on too much risk or concentrating in one place?

**The objective.** Each loan's expected return over the 7-year window:

```
expected return = (1 - PD) x interest_income_7yr  -  PD x loss_if_default
```

**Fractional formulation.** Each loan gets a variable between 0 and 1, the share of it we fund. That is realistic, since mortgage participations let a lender fund a fraction of a pool, and it keeps this a pure LP that solves in seconds instead of a mixed-integer problem that can stall.

---

## Setup and constraint parameters

Loads the pool, confirms it survived the handoff from the ML notebook, and sets the numbers the LP runs on.

**Everything is measured in dollars.** Budget, state cap, and all three floors. A lender's exposure is capital, not headcount. Mixing units would also make the shadow prices incomparable, since each would answer a different question.

**Budget: 10% of pool UPB.** Funding the whole pool is not a decision. A tenth forces real selection.

**Average PD ceiling: the pool's own average, 3.4016%.** Computed from the data rather than picked, so the requirement is that the funded book be no riskier than the pool it came from. It does not bind. Chasing return already produces a portfolio around 2.49%, nearly a point safer, because defaults cost more than the extra yield on risky loans is worth.

**State cap: 8% of budget.** California is 19.5% of pool dollars, the only state whose share alone forces the cap to bite. Texas is next at 7.4% and sits just under.

**Floors at each program's natural pool share.** First-time 20%, HomeReady 5.05%, HFA 2.56%. A floor at the natural share holds the portfolio where the pool already sits, so it binds whenever the optimizer would otherwise drift below. That is the behavior we want to observe before sweeping the values upward. Raising them is a policy question the floor sweep answers with numbers.

Sources: OCC Comptroller's Handbook on Concentrations of Credit for the state cap, FHFA affordable housing goals at 12 CFR Part 1282 and CRA for the floors.

In [15]:
import polars as pl
import numpy as np
import time
from pathlib import Path

pl.Config.set_tbl_rows(200)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(200)

PROC = Path("../data/processed")
pool = pl.read_parquet(PROC / "prod_pool.parquet")

print(f"loans   : {len(pool):,}")
print(f"columns : {len(pool.columns)}")
print(f"nulls   : {sum(pool[c].null_count() for c in pool.columns)}")

# ---------------- constraint parameters, all in dollars ----------------
BUDGET_FRAC     = 0.10      # of total pool UPB
PD_CEILING      = 0.030     # average PD across the funded portfolio
STATE_CAP       = 0.06      # any one state, share of budget
FT_FLOOR        = 0.20      # first-time buyer
HOMEREADY_FLOOR = 0.0505    # HomeReady, at its natural pool share
HFA_FLOOR       = 0.0256    # HFA, at its natural pool share

total_upb = pool["ORIG_UPB"].sum()
budget = BUDGET_FRAC * total_upb

print(f"\n--- budget ---")
print(f"  total pool UPB : ${total_upb/1e9:,.2f}B")
print(f"  budget ({BUDGET_FRAC:.0%})   : ${budget/1e9:,.3f}B")

print(f"\n--- constraints ---")
print(f"  avg PD ceiling   <= {PD_CEILING:.1%}")
print(f"  state cap        <= {STATE_CAP:.0%} of budget  "
      f"(${STATE_CAP*budget/1e9:.3f}B per state)")
print(f"  first-time floor >= {FT_FLOOR:.2%} of budget  "
      f"(${FT_FLOOR*budget/1e9:.3f}B)")
print(f"  HomeReady floor  >= {HOMEREADY_FLOOR:.2%} of budget  "
      f"(${HOMEREADY_FLOOR*budget/1e9:.3f}B)")
print(f"  HFA floor        >= {HFA_FLOOR:.2%} of budget  "
      f"(${HFA_FLOOR*budget/1e9:.3f}B)")

# ---------------- the objective vector ----------------
pool = pool.with_columns(
    ((1 - pl.col("pd")) * pl.col("interest_income_7yr")
     - pl.col("pd") * pl.col("loss_if_default")).alias("exp_return")
)

print(f"\n--- expected return per loan ---")
print(f"  total if we funded everything : ${pool['exp_return'].sum()/1e9:,.2f}B")
print(f"  mean                          : ${pool['exp_return'].mean():,.0f}")
print(f"  min                           : ${pool['exp_return'].min():,.0f}")
print(f"  max                           : ${pool['exp_return'].max():,.0f}")
print(f"  loans with negative return    : {(pool['exp_return'] < 0).sum():,}")

# ---------------- policy flags: is each floor reachable ----------------
print(f"\n--- policy programs in the pool ---")
print(f"{'program':16} {'loans':>9} {'UPB $B':>9} {'pool share':>11} "
      f"{'floor':>8} {'need':>9} {'default':>9}")
print("-" * 76)
for flag, floor, label in [("is_first_time", FT_FLOOR, "first-time"),
                           ("is_homeready", HOMEREADY_FLOOR, "HomeReady"),
                           ("is_hfa", HFA_FLOOR, "HFA")]:
    sub = pool.filter(pl.col(flag) == 1)
    avail = sub["ORIG_UPB"].sum()
    need = floor * budget
    print(f"{label:16} {len(sub):>9,} {avail/1e9:>9.2f} "
          f"{avail/total_upb:>10.2%} {floor:>8.2%} "
          f"{need/avail:>8.1%} {sub['default_flag'].mean():>8.2%}")
print("-" * 76)
print(f"  'need' = share of that program's available dollars the floor forces us to fund")
print(f"  book default rate: {pool['default_flag'].mean():.2%}")

# ---------------- state concentration ----------------
by_state = (
    pool.group_by("STATE")
        .agg(
            pl.len().alias("loans"),
            pl.col("ORIG_UPB").sum().alias("upb"),
            pl.col("pd").mean().alias("mean_pd"),
            pl.col("default_flag").mean().alias("actual_rate"),
        )
        .with_columns((pl.col("upb") / total_upb).alias("upb_share"))
        .sort("upb", descending=True)
)

print(f"\n--- top 10 states by dollars ---")
print(
    by_state.head(10).select(
        "STATE", "loans",
        (pl.col("upb") / 1e9).round(2).alias("upb_B"),
        (pl.col("upb_share") * 100).round(2).alias("pct_of_pool"),
        (pl.col("mean_pd") * 100).round(2).alias("pred_pd_pct"),
        (pl.col("actual_rate") * 100).round(2).alias("actual_pct"),
    )
)

over = by_state.filter(pl.col("upb_share") > STATE_CAP)
print(f"\n  states whose pool share alone exceeds the {STATE_CAP:.0%} cap: {len(over)}")
if len(over):
    print(over.select("STATE", (pl.col("upb_share") * 100).round(2).alias("pct_of_pool")))
print(f"  states in the pool: {len(by_state)}")

loans   : 409,857
columns : 21
nulls   : 0

--- budget ---
  total pool UPB : $93.76B
  budget (10%)   : $9.376B

--- constraints ---
  avg PD ceiling   <= 3.0%
  state cap        <= 6% of budget  ($0.563B per state)
  first-time floor >= 20.00% of budget  ($1.875B)
  HomeReady floor  >= 5.05% of budget  ($0.473B)
  HFA floor        >= 2.56% of budget  ($0.240B)

--- expected return per loan ---
  total if we funded everything : $23.02B
  mean                          : $56,154
  min                           : $-23,246
  max                           : $378,383
  loans with negative return    : 5

--- policy programs in the pool ---
program              loans    UPB $B  pool share    floor      need   default
----------------------------------------------------------------------------
first-time          97,499     21.87     23.32%   20.00%     8.6%    4.77%
HomeReady           20,682      3.77      4.02%    5.05%    12.6%    5.86%
HFA                 10,510      1.81      1.93%    2.

## Budget-only solve

One variable per loan, the objective, and the budget. Nothing else.

**Why this comes first.** With only a budget constraint, this is a fractional knapsack, and sorting by return per dollar is provably optimal for that problem. So we can check Gurobi's answer against a five-line sort. If they match, the model is wired correctly (objective sign, budget units, solver running) before we add the constraints that actually matter.

**It also sets the ceiling.** Every constrained run gets measured against this number. The difference is what the constraints cost.

**Fractional, not binary.** Each variable sits anywhere between 0 and 1. LP theory says the number of partial loans at an optimal solution is bounded by the number of constraints, so with one constraint we should see at most one partial loan and the rest at exactly 0 or 1. Checked below.

In [16]:
import gurobipy as gp
from gurobipy import GRB

upb = pool["ORIG_UPB"].to_numpy()
c = pool["exp_return"].to_numpy()
n = len(pool)

m = gp.Model("budget_only")
m.Params.OutputFlag = 0

x = m.addMVar(n, lb=0.0, ub=1.0, name="x")
m.setObjective(c @ x, GRB.MAXIMIZE)
m.addConstr(upb @ x <= budget, name="budget")

t = time.time()
m.optimize()
solve_time = time.time() - t

x_budget = x.X
funded_upb = upb @ x_budget
obj_budget = m.ObjVal

print(f"status          : {m.Status}  (2 = optimal)")
print(f"solve time      : {solve_time:.1f}s")
print(f"\nobjective       : ${obj_budget/1e9:,.3f}B expected return")
print(f"budget used     : ${funded_upb/1e9:,.3f}B of ${budget/1e9:,.3f}B")
print(f"return on funded: {obj_budget/funded_upb*100:.2f}%")

print(f"\nloans funded    : {(x_budget > 1e-6).sum():,} full or partial")
print(f"  fully (x=1)   : {(x_budget > 1-1e-6).sum():,}")
print(f"  partial       : {((x_budget > 1e-6) & (x_budget < 1-1e-6)).sum():,}")
print(f"  not funded    : {(x_budget <= 1e-6).sum():,}")

# ---- check Gurobi against the greedy sort ----
# Fractional knapsack: sort by return per dollar, fill until the money runs out.
ratio = c / upb
order = np.argsort(-ratio)
cum = np.cumsum(upb[order])
cutoff = np.searchsorted(cum, budget)

x_greedy = np.zeros(n)
x_greedy[order[:cutoff]] = 1.0
if cutoff < n:
    spent = cum[cutoff - 1] if cutoff > 0 else 0.0
    x_greedy[order[cutoff]] = (budget - spent) / upb[order[cutoff]]

obj_greedy = c @ x_greedy

print(f"\n--- does greedy-by-return-per-dollar match Gurobi ---")
print(f"  Gurobi objective : ${obj_budget/1e9:,.6f}B")
print(f"  greedy objective : ${obj_greedy/1e9:,.6f}B")
print(f"  difference       : ${abs(obj_budget - obj_greedy):,.2f}")
print(f"  same loans funded: {(np.abs(x_budget - x_greedy) < 1e-6).sum():,} of {n:,}")

# ---- what the unconstrained portfolio looks like ----
funded = pool.with_columns(pl.Series("x", x_budget)).filter(pl.col("x") > 1e-6)
w = funded["x"].to_numpy() * funded["ORIG_UPB"].to_numpy()

print(f"\n--- where the budget-only portfolio lands on its own ---")
print(f"  average PD    : {(funded['pd'].to_numpy() * w).sum() / w.sum():.4%}"
      f"   (ceiling will be {PD_CEILING:.1%})")
print(f"  first-time    : {w[funded['is_first_time'].to_numpy() == 1].sum() / budget:.2%}"
      f"   (floor {FT_FLOOR:.2%})")
print(f"  HomeReady     : {w[funded['is_homeready'].to_numpy() == 1].sum() / budget:.2%}"
      f"   (floor {HOMEREADY_FLOOR:.2%})")
print(f"  HFA           : {w[funded['is_hfa'].to_numpy() == 1].sum() / budget:.2%}"
      f"   (floor {HFA_FLOOR:.2%})")

top_state = (
    funded.with_columns(pl.Series("dollars", w))
          .group_by("STATE").agg(pl.col("dollars").sum())
          .with_columns((pl.col("dollars") / budget).alias("share"))
          .sort("share", descending=True)
)
print(f"\n  top 5 states by share of budget (cap will be {STATE_CAP:.0%}):")
print(top_state.head(5).select("STATE",
      (pl.col("share") * 100).round(2).alias("pct_of_budget")))

status          : 2  (2 = optimal)
solve time      : 0.5s

objective       : $2.826B expected return
budget used     : $9.376B of $9.376B
return on funded: 30.14%

loans funded    : 44,734 full or partial
  fully (x=1)   : 44,733
  partial       : 1
  not funded    : 365,123

--- does greedy-by-return-per-dollar match Gurobi ---
  Gurobi objective : $2.825826B
  greedy objective : $2.825826B
  difference       : $0.00
  same loans funded: 409,857 of 409,857

--- where the budget-only portfolio lands on its own ---
  average PD    : 2.4988%   (ceiling will be 3.0%)
  first-time    : 19.17%   (floor 20.00%)
  HomeReady     : 1.80%   (floor 5.05%)
  HFA           : 4.30%   (floor 2.56%)

  top 5 states by share of budget (cap will be 6%):
shape: (5, 2)
┌───────┬───────────────┐
│ STATE ┆ pct_of_budget │
│ ---   ┆ ---           │
│ str   ┆ f64           │
╞═══════╪═══════════════╡
│ CA    ┆ 28.16         │
│ AZ    ┆ 6.68          │
│ WA    ┆ 5.43          │
│ CO    ┆ 5.36          │
│ TX  

## Finding: the model is wired right, and we can see which constraints will bind

**The check passed exactly.** Gurobi and a greedy sort by return per dollar produced the same objective to the dollar and funded the identical 409,857 variables. With one constraint, LP theory says at most one loan comes back partial, and exactly one did. The objective sign, the budget units, and the solver are all correct.

**The return ceiling:**

- $2.826B expected return on $9.376B funded
- 30.14% return on funded dollars over the 7-year window
- 44,734 loans funded, about 11% of the pool

Every constrained run gets measured against this. The difference is what the constraints cost.

The scaffold's budget-only run came in at $2.828B and 30.15%. Essentially unchanged, so the tuned CatBoost and the move from 23 features to 18 did not shift the ceiling.

**Where the unconstrained portfolio lands on its own:**

| | Lands at | Constraint | Will it bind |
|---|---|---|---|
| Average PD | 2.4988% | pool average, 3.4016% | No, well under |
| First-time | 19.17% | floor 20.00% | Barely |
| HomeReady | 1.80% | floor 5.05% | Yes, hard |
| HFA | 4.30% | floor 2.56% | No, already above |
| California | 28.16% | cap 6% | Yes, hard |
| Arizona | 6.68% | cap 6% | Yes |

Three things stand out.

- **California goes to 28.16% when nothing stops it.** The pool is only 19.52% California, so the optimizer loads up well beyond the state's natural weight. California loans are unusually good on return per dollar. The cap has to cut that position by more than three quarters, which is why it is the most expensive state constraint.
- **HFA lands at 4.30% without being asked.** Those loans default at 9.15%, nearly 3x the book, and the optimizer buys them anyway. The rate on that paper is paying for the risk. Its floor at 2.56% does nothing.
- **HomeReady is the opposite.** The optimizer takes only 1.80% on its own against a 5.05% floor, so that floor does real work. Same program family, opposite behavior, and worth explaining in the report.

**On the average PD ceiling.** Chasing return already produces a book at 2.4988%, almost a full point safer than the pool it came from. Nobody told the optimizer to be safe. It got there because defaults cost more than the extra yield on risky loans is worth. Setting the ceiling at the pool average and finding it does not bind is a result, not a wasted constraint.

**One caveat on reading this table.** These are the shares the optimizer picks with only a budget in play. Adding the constraints changes which loans get funded, so a program sitting above its floor here can still shift once the state caps start moving money around. The solve is what settles it.

## Full solve: all constraints on

Same objective and budget, plus the average PD ceiling, a cap on every state, and the three program floors.

**The PD ceiling is the pool's own average**, computed from the data rather than typed in. The requirement is that the funded book be no riskier than the pool it was drawn from.

**The state cap applies to all 54 states**, not just California. Only California's pool share exceeds it outright, but the optimizer could concentrate anywhere, so every state gets the constraint and the shadow prices tell us which ones actually bit.

**What we read off the solve**

- What the constraints cost against the $2.826B budget-only ceiling
- The shadow price on each constraint, which is what one more dollar of room would be worth. Zero means the constraint is not binding.
- How many loans came back partial. LP theory bounds that by the number of constraints.

In [17]:
# the ceiling is the pool's own average PD, not a chosen round number
PD_CEILING = float(pool["pd"].mean())
print(f"average PD ceiling set to the pool average: {PD_CEILING:.4%}\n")

pd_vals = pool["pd"].to_numpy()
states = pool["STATE"].to_numpy()
state_list = sorted(set(states))

m = gp.Model("full_lp")
m.Params.OutputFlag = 0

x = m.addMVar(n, lb=0.0, ub=1.0, name="x")
m.setObjective(c @ x, GRB.MAXIMIZE)

con = {}
con["budget"] = m.addConstr(upb @ x <= budget, name="budget")

# average PD, written as a dollar-weighted constraint so it stays linear
con["pd_ceiling"] = m.addConstr(
    (pd_vals - PD_CEILING) * upb @ x <= 0, name="pd_ceiling")

for s in state_list:
    mask = (states == s).astype(float)
    con[f"state_{s}"] = m.addConstr(
        (upb * mask) @ x <= STATE_CAP * budget, name=f"state_{s}")

for flag, floor, label in [("is_first_time", FT_FLOOR, "first_time"),
                           ("is_homeready", HOMEREADY_FLOOR, "homeready"),
                           ("is_hfa", HFA_FLOOR, "hfa")]:
    mask = pool[flag].to_numpy().astype(float)
    con[f"floor_{label}"] = m.addConstr(
        (upb * mask) @ x >= floor * budget, name=f"floor_{label}")

t = time.time()
m.optimize()
print(f"status     : {m.Status}  (2 = optimal)")
print(f"solve time : {time.time()-t:.1f}s")
print(f"constraints: {m.NumConstrs}")

x_full = x.X
obj_full = m.ObjVal
funded_upb_full = upb @ x_full

print(f"\n--- result ---")
print(f"  objective        : ${obj_full/1e9:,.3f}B")
print(f"  budget-only was  : ${obj_budget/1e9:,.3f}B")
print(f"  constraints cost : ${(obj_budget-obj_full)/1e6:,.1f}M  "
      f"({(obj_budget-obj_full)/obj_budget:.2%})")
print(f"  budget used      : ${funded_upb_full/1e9:,.3f}B")
print(f"  return on funded : {obj_full/funded_upb_full*100:.2f}%")

print(f"\n  loans funded : {(x_full > 1e-6).sum():,}")
print(f"    fully      : {(x_full > 1-1e-6).sum():,}")
print(f"    partial    : {((x_full > 1e-6) & (x_full < 1-1e-6)).sum():,}")

# ---- which constraints bind ----
binding = [(name, cn.Pi) for name, cn in con.items() if abs(cn.Pi) > 1e-9]
binding.sort(key=lambda kv: -abs(kv[1]))

print(f"\n--- binding constraints: {len(binding)} of {m.NumConstrs} ---")
print(f"{'constraint':22} {'shadow price':>14}")
print("-" * 38)
for name, pi in binding:
    print(f"{name:22} {pi:>+14.4f}")
print("-" * 38)
print("  shadow price = extra return from one more dollar of room")
print("  positive on a ceiling means it is holding us back")
print("  negative on a floor means it is forcing us to give up return")

# ---- where the portfolio landed ----
funded_f = pool.with_columns(pl.Series("x", x_full)).filter(pl.col("x") > 1e-6)
wf = funded_f["x"].to_numpy() * funded_f["ORIG_UPB"].to_numpy()

print(f"\n--- where the constrained portfolio landed ---")
print(f"{'':14} {'budget-only':>13} {'constrained':>13} {'limit':>10}")
print("-" * 54)
print(f"{'average PD':14} {2.4988:>12.4f}% "
      f"{(funded_f['pd'].to_numpy()*wf).sum()/wf.sum()*100:>12.4f}% "
      f"{PD_CEILING*100:>9.4f}%")
for flag, floor, label, before in [
        ("is_first_time", FT_FLOOR, "first-time", 19.17),
        ("is_homeready", HOMEREADY_FLOOR, "HomeReady", 1.80),
        ("is_hfa", HFA_FLOOR, "HFA", 4.30)]:
    now = wf[funded_f[flag].to_numpy() == 1].sum() / budget * 100
    print(f"{label:14} {before:>12.2f}% {now:>12.2f}% {floor*100:>9.2f}%")

st = (
    funded_f.with_columns(pl.Series("dollars", wf))
            .group_by("STATE").agg(pl.col("dollars").sum())
            .with_columns((pl.col("dollars")/budget*100).alias("pct_of_budget"))
            .sort("pct_of_budget", descending=True)
)
print(f"\n  top 5 states, cap is {STATE_CAP*100:.0f}%:")
print(st.head(5).select("STATE", pl.col("pct_of_budget").round(2)))

average PD ceiling set to the pool average: 3.4016%

status     : 2  (2 = optimal)
solve time : 1.9s
constraints: 59

--- result ---
  objective        : $2.797B
  budget-only was  : $2.826B
  constraints cost : $28.7M  (1.02%)
  budget used      : $9.376B
  return on funded : 29.83%

  loans funded : 49,366
    fully      : 49,358
    partial    : 8

--- binding constraints: 8 of 59 ---
constraint               shadow price
--------------------------------------
budget                        +0.2814
state_CA                      +0.0291
floor_homeready               -0.0114
state_AZ                      +0.0077
state_WA                      +0.0039
state_CO                      +0.0037
state_TX                      +0.0030
state_FL                      +0.0015
--------------------------------------
  shadow price = extra return from one more dollar of room
  positive on a ceiling means it is holding us back
  negative on a floor means it is forcing us to give up return

--- where the 

## Finding: 6% is where the state cap earns its keep

Three runs, everything else held fixed.

| Cap | Objective | Constraints cost | Cost of tightening | Binding | Binding states |
|---|---|---|---|---|---|
| 8% | $2.803B | $22.6M (0.80%) | — | 4 | CA, AZ |
| **6%** | **$2.797B** | **$28.7M (1.02%)** | **+$6.1M** | **8** | **CA, AZ, WA, CO, TX, FL** |
| 5% | $2.791B | $34.5M (1.22%) | +$5.8M | 8 | same six |

**The cost curve is flat.** About $3M per percentage point, no cliff anywhere. The pool is not running short of good loans outside the concentrated states, so the cap could tighten further without breaking anything.

**The binding set stops growing at 6%.** Going from 8% to 6% pulled in four more states. Going from 6% to 5% pulled in none. Below 6% the cap is squeezing states that are already capped rather than catching new concentration.

**So 6% is the setting.** It catches every state that would otherwise concentrate, at the lowest cost that does so. The extra $5.8M for 5% buys no additional diversification. That reasoning is independent of anything the simulation might show later, which is the point.

**Shadow prices at 6%:**

| Constraint | Shadow price |
|---|---|
| Budget | +0.2814 |
| California | +0.0291 |
| HomeReady floor | -0.0114 |
| Arizona | +0.0077 |
| Washington | +0.0039 |
| Colorado | +0.0037 |
| Texas | +0.0030 |
| Florida | +0.0015 |

The budget dominates everything, as expected when almost every loan is profitable. California is the expensive state constraint by a factor of four over the next one: another dollar of California room would return 2.9 extra cents.

California's shadow price rises as the cap tightens (+0.0241 at 8%, +0.0291 at 6%, +0.0329 at 5%). Each remaining California dollar gets more valuable as you squeeze, which is the direction the model should move and a check that it is behaving.

**The HomeReady floor is the only floor doing anything.** At -0.0114, each dollar forced into HomeReady gives up about 1.1 cents. First-time lands at 21.97% on its own against a 20% floor, and HFA at 4.47% against a 2.56% floor. Neither binds.

**The average PD ceiling does not bind at any cap setting.** Set at the pool's own average of 3.4016%, the portfolio comes in at 2.4860%, nearly a full point safer. Chasing return already produces a book less risky than the pool it was drawn from, even with six states capped and money pushed into less-preferred geographies. Nobody told the optimizer to be safe. Defaults simply cost more than the extra yield on risky loans is worth.

**Partial loans track binding constraints exactly.** Four binding gave four partials at 8%, eight gave eight at 6% and 5%. LP theory says the number of fractional loans at an optimal solution is bounded by the number of binding constraints, and it held on every run.

## Is the solution clean?

The numbers look right. This checks whether to trust them.

**Three things it looks at**

- **The partial loans.** LP theory bounds the number of fractional loans by the number of binding constraints. We got 8 and 8, which matches, but we should see the actual values. A partial at 0.9999 would suggest a tolerance problem rather than a real fractional solution.
- **Slack on every constraint.** How close did the non-binding ones come? A constraint sitting at 99% of its limit with a shadow price of zero is a different situation from one sitting at 40%.
- **What the funded portfolio actually looks like.** Credit grade mix, PD spread, loan size, and return, compared against the pool it came from. If the optimizer is doing something sensible, the funded book should be visibly better on risk and return than the average loan.

In [20]:
# ---- 1. the partial loans ----
part_mask = (x_full > 1e-6) & (x_full < 1 - 1e-6)
print(f"--- partial loans: {int(part_mask.sum())} ---")
if part_mask.sum():
    print(
        pool.with_columns([
                pl.Series("x", x_full),
                pl.Series("_partial", part_mask),
            ])
            .filter(pl.col("_partial"))
            .select("LOAN_ID", "STATE", "credit_grade",
                    pl.col("x").round(6),
                    (pl.col("ORIG_UPB") / 1e3).round(0).alias("upb_k"),
                    (pl.col("pd") * 100).round(2).alias("pd_pct"),
                    (pl.col("exp_return") / pl.col("ORIG_UPB") * 100)
                        .round(2).alias("return_pct"),
                    "is_homeready", "is_hfa")
            .sort("x")
    )
print(f"\nbinding constraints: {len(binding)}   partial loans: {int(part_mask.sum())}")
print("LP theory bounds partials by the number of binding constraints.")

# ---- 2. slack on every constraint ----
# Gurobi hands back numpy scalars here, so cast to plain float for Polars.
rows = [(name, float(cn.Slack), float(cn.Pi)) for name, cn in con.items()]
slack = pl.DataFrame(rows, schema=["constraint", "slack", "shadow_price"],
                     orient="row")

print("\n--- non-binding constraints, closest to their limit first ---")
print(
    slack.filter(pl.col("shadow_price").abs() < 1e-9)
         .with_columns((pl.col("slack").abs() / budget * 100)
                       .round(3).alias("slack_pct_of_budget"))
         .sort("slack_pct_of_budget")
         .head(8)
         .select("constraint", "slack_pct_of_budget")
)
print("  slack = room left before the constraint would bite, as % of budget")

# ---- 3. what the funded portfolio looks like ----
pool_w = pool["ORIG_UPB"].to_numpy()

def wavg(frame, col, weights):
    return float((frame[col].to_numpy() * weights).sum() / weights.sum())

print("\n--- funded portfolio vs the pool it came from ---")
print(f"{'':22} {'pool':>13} {'funded':>13}")
print("-" * 50)
print(f"{'loans':22} {len(pool):>13,} {len(funded_f):>13,}")
print(f"{'dollars':22} {pool_w.sum()/1e9:>12,.2f}B {wf.sum()/1e9:>12,.3f}B")
print(f"{'average PD':22} {wavg(pool,'pd',pool_w)*100:>12.4f}% "
      f"{wavg(funded_f,'pd',wf)*100:>12.4f}%")
print(f"{'actual default rate':22} {wavg(pool,'default_flag',pool_w)*100:>12.4f}% "
      f"{wavg(funded_f,'default_flag',wf)*100:>12.4f}%")
print(f"{'mean loan size':22} {pool['ORIG_UPB'].mean():>13,.0f} "
      f"{funded_f['ORIG_UPB'].mean():>13,.0f}")
print(f"{'mean rate':22} {pool['ORIG_RATE'].mean():>12.3f}% "
      f"{funded_f['ORIG_RATE'].mean():>12.3f}%")
print(f"{'return per dollar':22} "
      f"{pool['exp_return'].sum()/pool_w.sum()*100:>12.2f}% "
      f"{obj_full/wf.sum()*100:>12.2f}%")

print("\n--- credit grade mix, share of dollars ---")
print(
    pool.with_columns(pl.Series("x", x_full))
        .with_columns((pl.col("x") * pl.col("ORIG_UPB")).alias("funded_dollars"))
        .group_by("credit_grade")
        .agg(
            (pl.col("ORIG_UPB").sum() / pool_w.sum() * 100).round(2).alias("pool_pct"),
            (pl.col("funded_dollars").sum() / wf.sum() * 100).round(2).alias("funded_pct"),
            (pl.col("default_flag").mean() * 100).round(2).alias("default_pct"),
        )
        .sort("funded_pct", descending=True)
)

print("\n--- LTV band mix, share of dollars ---")
print(
    pool.with_columns(pl.Series("x", x_full))
        .with_columns((pl.col("x") * pl.col("ORIG_UPB")).alias("funded_dollars"))
        .group_by("ltv_bin")
        .agg(
            (pl.col("ORIG_UPB").sum() / pool_w.sum() * 100).round(2).alias("pool_pct"),
            (pl.col("funded_dollars").sum() / wf.sum() * 100).round(2).alias("funded_pct"),
            (pl.col("default_flag").mean() * 100).round(2).alias("default_pct"),
        )
        .sort("funded_pct", descending=True)
)

print("\n--- PD spread of funded loans ---")
print(funded_f["pd"].describe())

--- partial loans: 8 ---
shape: (8, 9)
┌──────────────┬───────┬──────────────┬──────────┬───────┬────────┬────────────┬──────────────┬────────┐
│ LOAN_ID      ┆ STATE ┆ credit_grade ┆ x        ┆ upb_k ┆ pd_pct ┆ return_pct ┆ is_homeready ┆ is_hfa │
│ ---          ┆ ---   ┆ ---          ┆ ---      ┆ ---   ┆ ---    ┆ ---        ┆ ---          ┆ ---    │
│ str          ┆ str   ┆ str          ┆ f64      ┆ f64   ┆ f64    ┆ f64        ┆ i8           ┆ i8     │
╞══════════════╪═══════╪══════════════╪══════════╪═══════╪════════╪════════════╪══════════════╪════════╡
│ 560831794520 ┆ FL    ┆ Good         ┆ 0.104329 ┆ 158.0 ┆ 6.23   ┆ 28.3       ┆ 0            ┆ 0      │
│ 916299431469 ┆ TX    ┆ Exceptional  ┆ 0.171627 ┆ 128.0 ┆ 1.91   ┆ 28.45      ┆ 0            ┆ 0      │
│ 452389285372 ┆ CA    ┆ Exceptional  ┆ 0.177979 ┆ 750.0 ┆ 0.42   ┆ 31.05      ┆ 0            ┆ 0      │
│ 435591991477 ┆ AZ    ┆ Very Good    ┆ 0.213427 ┆ 185.0 ┆ 2.55   ┆ 28.91      ┆ 0            ┆ 0      │
│ 291579536713 ┆

## Finding: the solution is clean, and the optimizer buys priced risk rather than safety

**The partials are real.** Eight binding constraints produced exactly eight fractional loans, with values from 0.104 to 0.767. Nothing sitting at 0.9999 or 0.0001, so these are genuine fractional solutions and not solver tolerance artifacts. LP theory bounds the number of partials by the number of binding constraints, and it held.

**The PD ceiling is the closest non-binding constraint**, with 0.916% of budget in slack. It does not bite, but it is nearer than any other inactive constraint. If the floors were raised or the state cap tightened further, this is the one that would activate next.

### The funded portfolio against the pool it came from

| | Pool | Funded |
|---|---|---|
| Loans | 409,857 | 49,366 |
| Dollars | $93.76B | $9.376B |
| Average predicted PD | 3.4958% | 2.4860% |
| Actual default rate | 3.4934% | 2.7902% |
| Mean loan size | $228,760 | $189,954 |
| Mean rate | 4.141% | 4.774% |
| Return per dollar | 24.55% | 29.83% |

**Lower risk and higher yield at the same time.** The funded book defaults at 2.79% against the pool's 3.49% while earning 63 basis points more in rate. That combination is the argument for the whole approach, and it shows up without being asked for.

**Predicted improvement overstates actual improvement.** Predicted PD fell 1.01 points, actual default rate fell 0.70. The selection was directionally right but the model thought it was avoiding more risk than it did. This is the first place in the project where predicted and actual visibly diverge, and it belongs in the report.

### What the optimizer actually prefers

**Credit grade.** Overweights Exceptional (14.7% to 16.6%) and Very Good (49.1% to 52.3%), cuts Fair/Poor nearly in half (6.1% to 3.8%). Sensible.

**LTV is not sensible in the same way.** It overweights the 70-80 band hard (39.5% to 51.2%) and also overweights above 95 (4.6% to 7.8%), even though those loans default at 6.22%, nearly double the book. Meanwhile it cuts the safest band below 60 by more than half (16.8% to 8.0%).

**That is the pattern worth naming.** The optimizer is not avoiding risk. It is buying risk that is priced correctly and declining safety that is priced expensively. Low-LTV borrowers get the best rates, so their loans yield too little to earn a place. High-LTV borrowers pay enough extra to cover their higher default rate and still clear the bar.

The same thing shows in the HFA program, which lands at 4.47% of budget against a 2.56% floor despite defaulting at 9.15%. Nobody forced those loans in. The rate on that paper pays for the risk.

**Mean loan size fell by $39,000.** Smaller loans return more per dollar, so the budget stretches to more of them.

## Floor sweep: what does each policy floor cost?

Raise one floor at a time, hold the other two at base, and record the return given up. That turns "should we raise the floors" into a price list.

**Same ranges as the scaffold** so the numbers are comparable: first-time to 45%, HomeReady to 15%, HFA to 8%.

**Two things to watch this time.**

- The state cap is now 6% instead of 8%, and six states bind instead of two. The floors have less room to move money around, so costs should come in higher than the scaffold's.
- The average PD ceiling has only 0.916% of budget in slack. Raising the floors pushes riskier loans in. If the ceiling starts binding partway up a sweep, the cost curve will bend, and that bend is worth reporting.

**Each run is a full solve**, so the cost is exact rather than estimated from shadow prices. Shadow prices only tell you the cost of the next dollar, and these are large moves.

In [21]:
def solve_lp(ft_floor, hr_floor, hfa_floor, state_cap=STATE_CAP,
             pd_ceiling=PD_CEILING, report=False):
    """
    Full LP at the given constraint settings.
    Returns the objective plus which constraints bound.
    """
    mm = gp.Model()
    mm.Params.OutputFlag = 0
    xx = mm.addMVar(n, lb=0.0, ub=1.0)
    mm.setObjective(c @ xx, GRB.MAXIMIZE)

    cc = {"budget": mm.addConstr(upb @ xx <= budget)}
    cc["pd_ceiling"] = mm.addConstr((pd_vals - pd_ceiling) * upb @ xx <= 0)

    for s in state_list:
        mask = (states == s).astype(float)
        cc[f"state_{s}"] = mm.addConstr((upb * mask) @ xx <= state_cap * budget)

    for flag, floor, label in [("is_first_time", ft_floor, "first_time"),
                               ("is_homeready", hr_floor, "homeready"),
                               ("is_hfa", hfa_floor, "hfa")]:
        mask = pool[flag].to_numpy().astype(float)
        cc[f"floor_{label}"] = mm.addConstr((upb * mask) @ xx >= floor * budget)

    mm.optimize()
    if mm.Status != GRB.OPTIMAL:
        return {"status": mm.Status, "obj": None}

    bound = [k for k, v in cc.items() if abs(v.Pi) > 1e-9]
    xv = xx.X
    w = xv * upb
    return {
        "status": mm.Status,
        "obj": mm.ObjVal,
        "binding": bound,
        "n_binding": len(bound),
        "pd_binds": "pd_ceiling" in bound,
        "avg_pd": float((pd_vals * w).sum() / w.sum()),
        "loans": int((xv > 1e-6).sum()),
    }

BASE = solve_lp(FT_FLOOR, HOMEREADY_FLOOR, HFA_FLOOR)
print(f"base objective: ${BASE['obj']/1e9:,.4f}B   "
      f"binding: {BASE['n_binding']}   avg PD: {BASE['avg_pd']:.4%}\n")

SWEEPS = {
    "first_time": (FT_FLOOR,        [0.25, 0.30, 0.35, 0.40, 0.45]),
    "homeready":  (HOMEREADY_FLOOR, [0.07, 0.09, 0.11, 0.13, 0.15]),
    "hfa":        (HFA_FLOOR,       [0.04, 0.05, 0.06, 0.07, 0.08]),
}

rows = []
t0 = time.time()
for name, (base_val, steps) in SWEEPS.items():
    for val in [base_val] + steps:
        kw = {"ft_floor": FT_FLOOR, "hr_floor": HOMEREADY_FLOOR,
              "hfa_floor": HFA_FLOOR}
        kw[{"first_time": "ft_floor", "homeready": "hr_floor",
            "hfa": "hfa_floor"}[name]] = val
        r = solve_lp(**kw)
        rows.append({
            "floor": name,
            "setting": val,
            "obj": r["obj"],
            "cost_M": (BASE["obj"] - r["obj"]) / 1e6 if r["obj"] else None,
            "avg_pd": r["avg_pd"],
            "n_binding": r["n_binding"],
            "pd_binds": r["pd_binds"],
            "loans": r["loans"],
        })
print(f"swept {len(rows)} settings in {(time.time()-t0)/60:.1f} min\n")

sweep = pl.DataFrame(rows)

for name in SWEEPS:
    print(f"--- {name} ---")
    print(
        sweep.filter(pl.col("floor") == name)
             .select(
                 (pl.col("setting") * 100).round(2).alias("floor_pct"),
                 pl.col("cost_M").round(1).alias("cost_$M"),
                 (pl.col("avg_pd") * 100).round(4).alias("avg_pd_pct"),
                 pl.col("n_binding"),
                 pl.col("pd_binds").alias("pd_ceiling_binds"),
                 pl.col("loans"),
             )
    )
    print()

# ---- do the floors interact, or just add up ----
tops = {n: s[-1] for n, (b, s) in SWEEPS.items()}
all_up = solve_lp(tops["first_time"], tops["homeready"], tops["hfa"])
individual = sum(
    sweep.filter((pl.col("floor") == n) & (pl.col("setting") == v))["cost_M"][0]
    for n, v in tops.items()
)
combined = (BASE["obj"] - all_up["obj"]) / 1e6

print("--- all three floors at their highest tested setting ---")
print(f"  first-time {tops['first_time']:.0%}, HomeReady {tops['homeready']:.0%}, "
      f"HFA {tops['hfa']:.0%}")
print(f"  sum of the individual costs : ${individual:,.1f}M")
print(f"  actual combined cost        : ${combined:,.1f}M")
print(f"  difference                  : ${individual - combined:+,.1f}M")
print(f"  binding constraints         : {all_up['n_binding']}   "
      f"PD ceiling binds: {all_up['pd_binds']}")
print(f"  average PD                  : {all_up['avg_pd']:.4%} "
      f"(ceiling {PD_CEILING:.4%})")
print(f"\n  cost against the ${BASE['obj']/1e9:,.3f}B base, on a "
      f"${budget/1e9:,.3f}B budget")

base objective: $2.7971B   binding: 8   avg PD: 2.4860%

swept 18 settings in 1.0 min

--- first_time ---
shape: (6, 6)
┌───────────┬─────────┬────────────┬───────────┬──────────────────┬───────┐
│ floor_pct ┆ cost_$M ┆ avg_pd_pct ┆ n_binding ┆ pd_ceiling_binds ┆ loans │
│ ---       ┆ ---     ┆ ---        ┆ ---       ┆ ---              ┆ ---   │
│ f64       ┆ f64     ┆ f64        ┆ i64       ┆ bool             ┆ i64   │
╞═══════════╪═════════╪════════════╪═══════════╪══════════════════╪═══════╡
│ 20.0      ┆ 0.0     ┆ 2.486      ┆ 8         ┆ false            ┆ 49366 │
│ 25.0      ┆ 0.4     ┆ 2.4887     ┆ 9         ┆ false            ┆ 49338 │
│ 30.0      ┆ 2.6     ┆ 2.5018     ┆ 9         ┆ false            ┆ 49358 │
│ 35.0      ┆ 6.5     ┆ 2.5052     ┆ 9         ┆ false            ┆ 49234 │
│ 40.0      ┆ 12.0    ┆ 2.5153     ┆ 9         ┆ false            ┆ 49109 │
│ 45.0      ┆ 19.1    ┆ 2.5199     ┆ 9         ┆ false            ┆ 48906 │
└───────────┴─────────┴────────────┴────────

## Finding: the floors are cheap, independent, and shaped very differently

Each floor swept alone, other two at base. Cost is return given up against the $2.797B base on a $9.376B budget.

| Floor raised to | First-time | HomeReady | HFA |
|---|---|---|---|
| Base | 20.00% — $0.0M | 5.05% — $0.0M | 2.56% — $0.0M |
| | 25.00% — $0.4M | 7.00% — $2.6M | 4.00% — $0.0M |
| | 30.00% — $2.6M | 9.00% — $6.1M | 5.00% — $0.1M |
| | 35.00% — $6.5M | 11.00% — $10.5M | 6.00% — $0.8M |
| | 40.00% — $12.0M | 13.00% — $15.7M | 7.00% — $2.1M |
| Highest tested | 45.00% — $19.1M | 15.00% — $21.5M | 8.00% — $4.1M |

**Three different shapes, three different reasons.**

- **First-time starts free, then accelerates.** Nothing until 25%, then the cost roughly doubles at each step. The optimizer lands at 21.97% on its own, so the first few points cost nothing. Past that it has to reach further down the return ranking with each step.
- **HomeReady costs from the first dollar and rises in a straight line**, about $2.7M per two points. It is the only floor already binding at base, because the optimizer takes just 1.80% of HomeReady when nothing forces it.
- **HFA is nearly free.** Zero all the way to 4%, and only $4.1M at 8%, more than triple its natural pool share. The optimizer already buys 4.47% of HFA voluntarily despite a 9.15% default rate, because the rate on that paper pays for the risk.

**The floors do not compete.** All three at their highest tested settings cost $36.6M. The individual costs sum to $44.7M. They overlap by $8.1M, because some loans count toward more than one floor at once.

**The PD ceiling holds even at the extreme.** With all three floors maxed, the average PD reaches 2.7632%, still 0.64 points under the 3.4016% ceiling. So no plausible floor setting makes the portfolio riskier than the pool it came from. The ceiling never binds anywhere in this sweep.

**The tighter state cap did not make the floors more expensive.** The scaffold ran these same sweeps at an 8% state cap and got $19.4M, $22.0M, and $3.9M at the top settings. We get $19.1M, $21.5M, and $4.1M at 6%. The two constraints work on different things: the cap moves money across geography, the floors move it across programs.

**Loan counts move in opposite directions.** Raising HomeReady adds loans, 49,366 up to 50,096, because those loans are smaller and the budget stretches further. Raising first-time removes them, 49,366 down to 48,906, because it displaces small loans with larger ones.

**Total cost stays small.** Even the most aggressive setting tested, raising all three at once, costs $36.6M against a $2.797B portfolio. Under 1.3% of return. The equity-versus-return tradeoff is real but modest at this scale.

**What this does not answer.** These are expected returns on paper. Whether a portfolio carrying heavier equity-program exposure holds up differently in a bad year is a simulation question, and the raised floors have not been run through stage 4.

In [22]:
def real_return(x_vec):
    """
    What the portfolio actually earned, using observed defaults.

    A loan that paid returns its interest. A loan that defaulted costs
    its loss. Weighted by how much of the loan we funded.
    """
    d = pool["default_flag"].to_numpy()
    return float((x_vec * ((1 - d) * pool["interest_income_7yr"].to_numpy()
                           - d * pool["loss_if_default"].to_numpy())).sum())

def portfolio_stats(x_vec, label):
    w = x_vec * upb
    funded = w > 1e-6
    return {
        "portfolio": label,
        "expected_M": float((c * x_vec).sum()) / 1e6,
        "real_M": real_return(x_vec) / 1e6,
        "loans": int(funded.sum()),
        "avg_pd": float((pd_vals * w).sum() / w.sum()),
        "actual_default": float((pool["default_flag"].to_numpy() * w).sum() / w.sum()),
        "max_state": float(max(
            w[(states == s)].sum() for s in state_list) / budget),
        "first_time": float(w[pool["is_first_time"].to_numpy() == 1].sum() / budget),
        "homeready": float(w[pool["is_homeready"].to_numpy() == 1].sum() / budget),
        "hfa": float(w[pool["is_hfa"].to_numpy() == 1].sum() / budget),
    }

def fill_by(key, ascending):
    """Sort by a key and fund until the budget runs out, partial on the last."""
    order = np.argsort(key if ascending else -key)
    cum = np.cumsum(upb[order])
    cut = np.searchsorted(cum, budget)
    x_vec = np.zeros(n)
    x_vec[order[:cut]] = 1.0
    if cut < n:
        spent = cum[cut - 1] if cut > 0 else 0.0
        x_vec[order[cut]] = (budget - spent) / upb[order[cut]]
    return x_vec

# --- objective vectors under each score ---
interest = pool["interest_income_7yr"].to_numpy()
loss = pool["loss_if_default"].to_numpy()
s_rule = pool["score_rule"].to_numpy()

c_cat  = (1 - pd_vals) * interest - pd_vals * loss      # CatBoost, already = c
c_rule = (1 - s_rule)  * interest - s_rule  * loss      # rule-based score

results = []
t0 = time.time()

for score_name, score, cvec in [("rule-based", s_rule, c_rule),
                                ("CatBoost", pd_vals, c_cat)]:
    # risk-sort: lowest predicted PD first
    results.append(portfolio_stats(fill_by(score, ascending=True),
                                   f"{score_name} / risk-sort"))
    # greedy-return: highest expected return per dollar first
    results.append(portfolio_stats(fill_by(cvec / upb, ascending=False),
                                   f"{score_name} / greedy-return"))

# LP under each score
mm = gp.Model(); mm.Params.OutputFlag = 0
xr = mm.addMVar(n, lb=0.0, ub=1.0)
mm.setObjective(c_rule @ xr, GRB.MAXIMIZE)
mm.addConstr(upb @ xr <= budget)
mm.addConstr((s_rule - float(s_rule.mean())) * upb @ xr <= 0)
for s in state_list:
    mask = (states == s).astype(float)
    mm.addConstr((upb * mask) @ xr <= STATE_CAP * budget)
for flag, floor in [("is_first_time", FT_FLOOR), ("is_homeready", HOMEREADY_FLOOR),
                    ("is_hfa", HFA_FLOOR)]:
    mask = pool[flag].to_numpy().astype(float)
    mm.addConstr((upb * mask) @ xr >= floor * budget)
mm.optimize()
results.append(portfolio_stats(xr.X, "rule-based / LP"))
results.append(portfolio_stats(x_full, "CatBoost / LP"))

print(f"built six portfolios in {time.time()-t0:.1f}s\n")

res = pl.DataFrame(results)
print("--- all six, real return using observed defaults ---")
print(
    res.select(
        "portfolio",
        pl.col("real_M").round(1).alias("real_$M"),
        pl.col("expected_M").round(1).alias("expected_$M"),
        "loans",
        (pl.col("avg_pd") * 100).round(3).alias("pred_pd_pct"),
        (pl.col("actual_default") * 100).round(3).alias("actual_pct"),
        (pl.col("max_state") * 100).round(1).alias("top_state_pct"),
    )
)

# --- the 2x3 grid ---
grid = {r["portfolio"]: r["real_M"] for r in results}
print("\n--- real return, $M ---")
print(f"{'':14} {'risk-sort':>14} {'greedy-return':>15} {'LP':>14}")
print("-" * 60)
for s in ["rule-based", "CatBoost"]:
    print(f"{s:14} {grid[f'{s} / risk-sort']:>14,.1f} "
          f"{grid[f'{s} / greedy-return']:>15,.1f} {grid[f'{s} / LP']:>14,.1f}")
print("-" * 60)

print("\n--- what a better score buys, holding the rule fixed ---")
for rule in ["risk-sort", "greedy-return", "LP"]:
    diff = grid[f"CatBoost / {rule}"] - grid[f"rule-based / {rule}"]
    print(f"  {rule:16} {diff:+9,.1f}M")

print("\n--- what the constraints cost, holding the score fixed ---")
for s in ["rule-based", "CatBoost"]:
    diff = grid[f"{s} / LP"] - grid[f"{s} / greedy-return"]
    print(f"  {s:16} {diff:+9,.1f}M")

built six portfolios in 3.2s

--- all six, real return using observed defaults ---
shape: (6, 7)
┌────────────────────────────┬─────────┬─────────────┬───────┬─────────────┬────────────┬───────────────┐
│ portfolio                  ┆ real_$M ┆ expected_$M ┆ loans ┆ pred_pd_pct ┆ actual_pct ┆ top_state_pct │
│ ---                        ┆ ---     ┆ ---         ┆ ---   ┆ ---         ┆ ---        ┆ ---           │
│ str                        ┆ f64     ┆ f64         ┆ i64   ┆ f64         ┆ f64        ┆ f64           │
╞════════════════════════════╪═════════╪═════════════╪═══════╪═════════════╪════════════╪═══════════════╡
│ rule-based / risk-sort     ┆ 2272.0  ┆ 2271.2      ┆ 43431 ┆ 0.881       ┆ 0.866      ┆ 22.0          │
│ rule-based / greedy-return ┆ 2723.8  ┆ 2759.3      ┆ 44523 ┆ 4.059       ┆ 4.658      ┆ 28.0          │
│ CatBoost / risk-sort       ┆ 2190.4  ┆ 2186.7      ┆ 44935 ┆ 0.337       ┆ 0.268      ┆ 12.7          │
│ CatBoost / greedy-return   ┆ 2806.9  ┆ 2825.8      ┆ 

## Finding: a better score only pays if the rule chases return

Real return in $M, scored on observed defaults.

| | risk-sort | greedy-return | LP |
|---|---|---|---|
| Rule-based | 2,272.0 | 2,723.8 | 2,707.0 |
| CatBoost | 2,190.4 | 2,806.9 | 2,778.8 |

**What the better score buys, holding the rule fixed:**

| Rule | CatBoost minus rule-based |
|---|---|
| risk-sort | **-$81.5M** |
| greedy-return | +$83.1M |
| LP | +$71.8M |

### The risk-sort result is the one worth explaining

CatBoost loses by $81.5M when the rule sorts by risk. That is not a failure of the model. It is the model working exactly as intended and the rule wasting it.

Look at what it produced: a 0.268% actual default rate, the safest portfolio of the six by a factor of three. CatBoost found genuinely safer loans than the lookup table could. And that portfolio earned the least of all six.

**Low PD comes with low rates, because good credit gets good pricing.** So the better your risk model, the more precisely you fund the lowest-yielding paper in the pool. Risk-sorting throws away return information entirely, and a sharper score just makes it throw it away more efficiently.

**This separates two things that would otherwise be tangled.** The value of the model is not that it identifies risk. The lookup table already does that reasonably well. The value is that it lets a return-seeking rule tell profitable risk apart from unprofitable risk, one loan at a time.

### Prediction quality shows up in the forecast miss

| Portfolio | Expected | Real | Miss |
|---|---|---|---|
| rule-based / greedy | $2,759.3M | $2,723.8M | -$35.5M |
| CatBoost / greedy | $2,825.8M | $2,806.9M | -$18.9M |
| rule-based / LP | $2,734.0M | $2,707.0M | -$27.0M |
| CatBoost / LP | $2,797.1M | $2,778.8M | -$18.3M |

Both scores understated defaults on the loans they selected. The rule-based score predicted 4.06% and got 4.66%. CatBoost predicted 2.50% and got 2.79%. CatBoost's forecast miss is about a third smaller in dollars.

That is the resolution advantage in a different form. Both scores are calibrated across the whole pool, but the funded portfolio is not the whole pool. It is a selected slice, and holding up on a selected slice is a harder test.

### What the constraints cost

| Score | LP minus greedy-return |
|---|---|
| Rule-based | -$16.8M |
| CatBoost | -$28.2M |

The real cost is larger than the $22.6M expected cost from the shadow prices, because the constraints move the portfolio into loans whose defaults the model had not priced as well.

**But greedy-return concentrates 28.2% of the budget in one state.** The LP holds 6%. On paper that concentration is worth $28.2M in giving it up. Whether it is worth it once a state-level shock exists is what stage 4 answers, and it is the first time the simulation will be able to see the difference.

## Save the portfolios

Last cell. Everything the simulation needs, plus the numbers the report will quote.

**`prod_portfolios.parquet`** carries one weight column per portfolio, not a list of funded loans. Portfolios have partial loans, and a list of LOAN_IDs would lose the fraction. `STATE` comes along because the simulation now has a state factor and needs to know where each loan sits.

**`prod_portfolios_summary.json`** holds the six portfolios' stats, the constraint settings, the shadow prices from the base solve, and the full floor sweep. The sweep goes in the file so the report can quote those numbers without rerunning it, which means the notebook and the report cannot drift apart.

In [23]:
import json
from datetime import datetime, timezone

# rebuild each portfolio's weight vector so all six land in one frame
x_vectors = {
    "rule_risk_sort":     fill_by(s_rule, ascending=True),
    "rule_greedy_return": fill_by(c_rule / upb, ascending=False),
    "rule_lp":            xr.X,
    "cat_risk_sort":      fill_by(pd_vals, ascending=True),
    "cat_greedy_return":  fill_by(c_cat / upb, ascending=False),
    "cat_lp":             x_full,
}

portfolios = pool.select(
    "LOAN_ID", "STATE", "credit_grade", "ltv_bin",
    "ORIG_UPB", "ORIG_TERM", "ORIG_RATE",
    "interest_income_7yr", "loss_if_default", "lgd",
    "is_first_time", "is_homeready", "is_hfa",
    "pd", "score_rule", "default_flag",
).with_columns([pl.Series(f"x_{k}", v) for k, v in x_vectors.items()])

portfolios.write_parquet(PROC / "prod_portfolios.parquet")
print(f"saved prod_portfolios.parquet   {portfolios.shape}")

summary = {
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "pool": {
        "file": "prod_pool.parquet",
        "loans": int(len(pool)),
        "total_upb": float(total_upb),
        "mean_pd": float(pd_vals.mean()),
        "actual_default_rate": float(pool["default_flag"].mean()),
    },
    "constraints": {
        "budget_frac": BUDGET_FRAC,
        "budget": float(budget),
        "pd_ceiling": float(PD_CEILING),
        "pd_ceiling_source": "the pool's own average PD, computed not chosen",
        "state_cap": STATE_CAP,
        "state_cap_source": "swept 8/6/5 percent; 6 catches every concentrating "
                            "state at the lowest cost that does so",
        "first_time_floor": FT_FLOOR,
        "homeready_floor": HOMEREADY_FLOOR,
        "hfa_floor": HFA_FLOOR,
        "floor_source": "each program's natural share of the pool",
    },
    "budget_only": {
        "objective": float(obj_budget),
        "loans": int((x_budget > 1e-6).sum()),
        "return_on_funded": float(obj_budget / (upb @ x_budget)),
        "matches_greedy_sort": bool(abs(obj_budget - obj_greedy) < 1.0),
    },
    "base_solve": {
        "objective": float(obj_full),
        "constraints_cost": float(obj_budget - obj_full),
        "constraints_cost_pct": float((obj_budget - obj_full) / obj_budget),
        "loans": int((x_full > 1e-6).sum()),
        "partial_loans": int(part_mask.sum()),
        "avg_pd": float((pd_vals * wf_all).sum() / wf_all.sum())
                  if (wf_all := x_full * upb).sum() else None,
        "binding": {name: float(pi) for name, pi in binding},
        "n_constraints": int(m.NumConstrs),
    },
    "portfolios": results,
    "floor_sweep": {
        "note": "each floor moved alone, other two at base; cost in dollars "
                "against the base objective",
        "base_objective": float(BASE["obj"]),
        "runs": sweep.to_dicts(),
        "all_three_maxed": {
            "settings": {k: float(v) for k, v in tops.items()},
            "sum_of_individual_costs_M": float(individual),
            "actual_combined_cost_M": float(combined),
            "overlap_M": float(individual - combined),
            "avg_pd": float(all_up["avg_pd"]),
            "pd_ceiling_binds": bool(all_up["pd_binds"]),
        },
    },
}

(PROC / "prod_portfolios_summary.json").write_text(json.dumps(summary, indent=2))
print("saved prod_portfolios_summary.json")

print("\n--- sanity ---")
print(f"  loans      : {len(portfolios):,}")
print(f"  nulls      : {sum(portfolios[c].null_count() for c in portfolios.columns)}")
print(f"  states     : {portfolios['STATE'].n_unique()}")
print(f"\n{'portfolio':22} {'funded $B':>11} {'loans':>9} {'real $M':>10}")
print("-" * 56)
for k, v in x_vectors.items():
    w = v * upb
    print(f"{k:22} {w.sum()/1e9:>11.3f} {int((v > 1e-6).sum()):>9,} "
          f"{real_return(v)/1e6:>10,.1f}")
print("-" * 56)
print(f"  budget: ${budget/1e9:.3f}B  (every portfolio should spend it all)")

saved prod_portfolios.parquet   (409857, 22)
saved prod_portfolios_summary.json

--- sanity ---
  loans      : 409,857
  nulls      : 0
  states     : 54

portfolio                funded $B     loans    real $M
--------------------------------------------------------
rule_risk_sort               9.376    43,431    2,272.0
rule_greedy_return           9.376    44,523    2,723.8
rule_lp                      9.376    48,397    2,707.0
cat_risk_sort                9.376    44,935    2,190.4
cat_greedy_return            9.376    44,734    2,806.9
cat_lp                       9.376    49,366    2,778.8
--------------------------------------------------------
  budget: $9.376B  (every portfolio should spend it all)


# Summary

**What this notebook produced.** Six portfolios, saved to `prod_portfolios.parquet`. That file is what the simulation reads.

---

### The setup

409,857 loans, $93.76B. Budget is 10% of that, $9.376B. Objective is each loan's expected return over 7 years: `(1 - PD) x interest - PD x loss`. Fractional formulation, so each loan gets a weight between 0 and 1.

**Constraints, and where the numbers came from:**

| Constraint | Setting | Source |
|---|---|---|
| Budget | 10% of pool UPB | forces real selection |
| Average PD ceiling | 3.4016% | the pool's own average, computed not chosen |
| State cap | 6% of budget | swept 8/6/5, see below |
| First-time floor | 20% | its natural pool share |
| HomeReady floor | 5.05% | its natural pool share |
| HFA floor | 2.56% | its natural pool share |

### Model verification

Budget-only, the LP is a fractional knapsack, and a greedy sort by return per dollar is provably optimal for that. Gurobi and the sort agreed to the dollar on all 409,857 variables. Objective sign, budget units, and solver all confirmed correct before the real constraints went on.

Budget-only ceiling: **$2.826B**, 30.14% on funded dollars, 44,734 loans.

### The base solve

**$2.797B**, 29.83% on funded dollars, 49,366 loans. Constraints cost $28.7M, 1.02%.

8 of 59 constraints bind: budget, California, Arizona, Washington, Colorado, Texas, Florida, and the HomeReady floor. Eight binding constraints produced exactly eight partial loans, which is the bound LP theory gives.

**The state cap was chosen by sweeping it.** At 8% only California and Arizona bound. At 6% all six concentrating states bound. At 5% the same six bound and it cost another $5.8M. So 6% catches every state that would otherwise concentrate, at the lowest cost that does so. That reasoning is independent of anything the simulation shows later.

**The PD ceiling never binds.** The portfolio lands at 2.4860%, nearly a point safer than the pool it came from. Nobody told the optimizer to be safe. Defaults simply cost more than the extra yield on risky loans is worth.

### The floor sweep

Written into `prod_portfolios_summary.json` so the report can quote it without rerunning.

| Floor raised to | First-time | HomeReady | HFA |
|---|---|---|---|
| Base | 20.00% — $0.0M | 5.05% — $0.0M | 2.56% — $0.0M |
| | 25.00% — $0.4M | 7.00% — $2.6M | 4.00% — $0.0M |
| | 30.00% — $2.6M | 9.00% — $6.1M | 5.00% — $0.1M |
| | 35.00% — $6.5M | 11.00% — $10.5M | 6.00% — $0.8M |
| | 40.00% — $12.0M | 13.00% — $15.7M | 7.00% — $2.1M |
| Highest tested | 45.00% — $19.1M | 15.00% — $21.5M | 8.00% — $4.1M |

Three different shapes. First-time is free until 25% then accelerates. HomeReady costs from the first dollar and rises linearly. HFA is nearly free, because the optimizer already buys 4.47% voluntarily despite a 9.15% default rate.

All three at their highest tested settings cost $36.6M against a sum of individual costs of $44.7M. They overlap by $8.1M, so the floors do not compete. Even then the average PD reaches only 2.7632%, still under the ceiling.

**Decision: floors stay at base.** The optimizer funds first-time buyers at 21.97% without being forced, which is a better finding than a floor that makes it happen. The sweep is the evidence if a policy recommendation is wanted.

### The six portfolios

Real return in $M, scored on observed defaults.

| | risk-sort | greedy-return | LP |
|---|---|---|---|
| Rule-based | 2,272.0 | 2,723.8 | 2,707.0 |
| CatBoost | 2,190.4 | 2,806.9 | 2,778.8 |

**A better score only pays if the rule chases return.** CatBoost wins by $83.1M on greedy-return and $71.8M on the LP, and loses by $81.5M on risk-sort.

That loss is the model working correctly and the rule wasting it. CatBoost/risk-sort produced a 0.268% actual default rate, three times safer than any other portfolio, and earned the least of all six. Low PD comes with low rates, because good credit gets good pricing. A sharper risk score just funds the lowest-yielding paper more precisely.

**CatBoost's forecast miss is about a third smaller.** Expected minus real is $18.3M for CatBoost/LP against $27.0M for rule-based/LP. Both scores are calibrated across the whole pool, but the funded portfolio is a selected slice, which is a harder test.

### The open question

The LP gives up $28.2M against greedy-return. In exchange, greedy holds 28.2% of the budget in one state and the LP holds 6%.

On expected return that trade looks like a loss. Whether it is a loss once a state-level shock exists is what stage 4 answers, and it is the first time the simulation will be able to see the difference.

### Files written

`prod_portfolios.parquet` (409,857 x 22) with a weight column per portfolio, and `prod_portfolios_summary.json` with the constraint settings, shadow prices, portfolio stats, and the full floor sweep.